# 02 - Feature Engineering

In [3]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/churn_data.csv")

# Same TotalCharges trap as 01_explore_data.ipynb: 11 brand-new customers
# (tenure == 0) store a blank-space string instead of a real NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df.shape

(7043, 21)

## `contract_risk`

In [4]:
contract_risk_map = {"Month-to-month": 2, "One year": 1, "Two year": 0}
df["contract_risk"] = df["Contract"].map(contract_risk_map)

df["contract_risk"].value_counts().sort_index()

contract_risk
0    1695
1    1473
2    3875
Name: count, dtype: int64

## `num_services`

In [5]:
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

df["num_services"].value_counts().sort_index()

num_services
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

## `charge_trend`

In [4]:
df["charge_trend"] = df["MonthlyCharges"] - (df["TotalCharges"] / df["tenure"])

print("NaN count:", df["charge_trend"].isna().sum())
df["charge_trend"].describe()

NaN count: 11


count    7032.000000
mean       -0.001215
std         2.616165
min       -18.900000
25%        -1.160179
50%         0.000000
75%         1.147775
max        19.125000
Name: charge_trend, dtype: float64

## `tenure` and `is_electronic_check`

In [5]:
df["is_electronic_check"] = (df["PaymentMethod"] == "Electronic check").astype(int)

df["is_electronic_check"].value_counts()

is_electronic_check
0    4678
1    2365
Name: count, dtype: int64

## Correlation with Churn

In [6]:
df["churn_flag"] = (df["Churn"] == "Yes").astype(int)

engineered = ["contract_risk", "num_services", "charge_trend", "tenure", "is_electronic_check"]

corr = df[engineered + ["churn_flag"]].corr()["churn_flag"].drop("churn_flag")
corr.sort_values(key=abs, ascending=False)

contract_risk          0.396713
tenure                -0.352229
is_electronic_check    0.301919
num_services          -0.087698
charge_trend           0.002160
Name: churn_flag, dtype: float64

In [7]:
top_feature = corr.abs().idxmax()
print(f"Strongest correlated feature: {top_feature} ({corr[top_feature]:.3f})")
assert top_feature == "contract_risk", "Expected contract_risk to be the strongest signal per the EDA sanity check"
print("Sanity check passed: contract_risk is the strongest signal, as expected from EDA.")

Strongest correlated feature: contract_risk (0.397)
Sanity check passed: contract_risk is the strongest signal, as expected from EDA.


## Handle missing values: `TotalCharges` (and `charge_trend`)

In [8]:
# Design decision: tenure == 0 rows get TotalCharges = 0 (true value, not an
# imputation) and charge_trend = 0 (no billing history yet => no trend).
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.loc[df["tenure"] == 0, "charge_trend"] = 0

print("TotalCharges NaNs:", df["TotalCharges"].isna().sum())
print("charge_trend NaNs:", df["charge_trend"].isna().sum())

TotalCharges NaNs: 0
charge_trend NaNs: 0


## One-hot encode

In [9]:
onehot_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "PaperlessBilling",
    "PaymentMethod",
]

df_encoded = pd.get_dummies(df, columns=onehot_cols, drop_first=True)
df_encoded.shape

(7043, 30)

## Drop non-feature columns

In [10]:
drop_cols = ["customerID", "Contract", "Churn"] + service_cols
df_model = df_encoded.drop(columns=drop_cols)

print("Final shape:", df_model.shape)
print("\nAny non-numeric columns left?")
print(df_model.dtypes[df_model.dtypes == "object"])
df_model.head()

Final shape: (7043, 21)

Any non-numeric columns left?
Series([], dtype: object)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,contract_risk,num_services,charge_trend,is_electronic_check,churn_flag,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,2,1,0.000000,1,0,False,True,False,False,True,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,1,2,1.376471,0,0,True,False,False,True,False,False,False,False,False,False,False,True
2,0,2,53.85,108.15,2,2,-0.225000,0,1,True,False,False,True,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,1,3,1.394444,0,0,True,False,False,False,True,False,False,False,False,False,False,False
4,0,2,70.70,151.65,2,0,-5.125000,1,1,False,False,False,True,False,False,True,False,True,False,True,False
